# فصل ۱۴: دانش خارجی و معماری RAG (External Knowledge & RAG)
## بخش ۲: الگوریتم‌های بازیابی - درس اول: بازیابی هوشمند با الگوریتم MMR و فیلتر آستانه شباهت

### 🎯 هدف درسنامه:
در درسنامه‌های پیشین یاد گرفتیم چگونه یک خط لوله RAG با قابلیت استناد به منابع بسازیم. اما یک چالش پنهان و بسیار رایج در دنیای واقعی وجود دارد:
**«آیا اسنادی که بازیاب به پرامپت تزریق می‌کند، واقعاً مفید و جامع هستند یا فقط چند بار یک حرف تکراری را با ادبیات مختلف بازگو می‌کنند؟»**

در این درسنامه با دو استراتژی کلیدی و حیاتی در موتورهای بازیاب (Retrievers) آشنا می‌شویم:
1. **الگوریتم MMR (Maximal Marginal Relevance):** ایجاد تعادل هوشمند بین **ارتباط (Relevance)** و **تنوع (Diversity)** برای نجات کانتکست از افزونگی (Redundancy).
2. **آستانه نمره شباهت (Score Thresholding):** جلوگیری از توهم مدل (Hallucination) در مواجهه با سوالات بی‌ربط و پرت (Out-of-Domain Queries).


### 🧩 مشکل بزرگ Similarity Search: دام تکرار و افزونگی (Redundancy)

فرض کنید یک سند طولانی درباره تاریخچه شرکت اپل داریم و کاربر این سوال دو بخشی را می‌پرسد:
> **«اپل چطور تاسیس شد و محصولات اولیه‌اش چه بود؟»**

در متن، ۵ بار در بخش‌های مختلف جمله *«استیو جابز اپل را در گاراژ خانه‌اش تاسیس کرد»* با کمی تغییر تکرار شده است:
- چانک ۱: *استیو جابز در گاراژ خانه‌اش در کالیفرنیا اپل را با همراهی وزنیاک تاسیس کرد...*
- چانک ۲: *در سال ۱۹۷۶ اپل در یک گاراژ محقر تاسیس شد و کار خود را آغاز نمود...*
- چانک ۳: *شروع کار و تاسیس رسمی اپل از گاراژ منزل پدری جابز کلید خورد...*
- چانک ۴: *اولین محصولات شرکت رایانه‌های Apple I و Apple II بودند که صنعت رایانه شخصی را متحول کردند...*

اگر از جستجوی شباهت ساده (`Similarity Search`) با `k=3` استفاده کنیم، چون چانک‌های ۱ و ۲ و ۳ بیشترین شباهت لغوی/معنایی را به واژه‌های «تاسیس» دارند، هر ۳ چانک انتخاب می‌شوند!
**نتیجه فاجعه‌بار:**
طول کانتکست مدل پر می‌شود، اما **هیچ اطلاعاتی درباره محصولات اولیه (چانک ۴) به مدل نمی‌رسد** و مدل فقط به نصف سوال کاربر پاسخ می‌دهد!

#### ⚽ مثال شهودی تیم فوتبال:
> برای ساختن بهترین تیم فوتبال دنیا، نمی‌توانید **۱۱ دروازه‌بان** استخدام کنید، حتی اگر آن ۱۱ نفر بهترین بازیکنان جهان باشند! شما به تعادل و تنوع نیاز دارید: دروازه‌بان، مدافع، هافبک و مهاجم.
> - **Similarity Search:** ۱۱ دروازه‌بان انتخاب می‌کند چون همه امتیاز بالایی دارند!
> - **MMR Search:** ابتدا بهترین دروازه‌بان را برمی‌دارد؛ اما برای نفر بعدی، تکراری‌ها را رد کرده و به سراغ مدافع و مهاجم می‌رود.


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

# فعال‌سازی حالت آفلاین برای مدل امبدینگ لوکال
os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# مدل امبدینگ چندزبانه سریع و دقیق
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# شبیه‌سازی اسناد تاریخچه اپل (دارای افزونگی و تنوع موضوعی)
apple_documents = [
    Document(
        page_content="استیو جابز به همراه استیو وزنیاک شرکت اپل را در گاراژ خانه پدری جابز در لوس آلتوس تاسیس کردند.",
        metadata={"source": "apple_history.txt", "topic": "founding", "id": 1}
    ),
    Document(
        page_content="در سال ۱۹۷۶ شرکت اپل به صورت رسمی در یک گاراژ محقر کار خود را آغاز و پایه‌گذاری شد.",
        metadata={"source": "apple_history.txt", "topic": "founding", "id": 2}
    ),
    Document(
        page_content="شروع به کار و تاسیس رسمی اپل در آوریل ۱۹۷۶ از گاراژ خانگی استیو جابز کلید خورد.",
        metadata={"source": "apple_history.txt", "topic": "founding", "id": 3}
    ),
    Document(
        page_content="نخستین محصولات تجاری شرکت کامپیوتر دست‌ساز Apple I و سپس کامپیوتر بسیار موفق Apple II بودند.",
        metadata={"source": "apple_products.txt", "topic": "products", "id": 4}
    ),
    Document(
        page_content="در سال ۱۹۸۴ اپل با معرفی رایانه مکینتاش (Macintosh) رابط کاربری گرافیکی را به بازار عرضه کرد.",
        metadata={"source": "apple_products.txt", "topic": "products", "id": 5}
    ),
    Document(
        page_content="شرکت اپل مشمول استانداردهای کنترل کیفیت سخت‌گیرانه بوده و خدمات پس از فروش جهانی ارائه می‌دهد.",
        metadata={"source": "apple_support.txt", "topic": "service", "id": 6}
    )
]

# ساخت و ذخیره‌سازی در پایگاه داده برداری کروما
vector_db = Chroma.from_documents(
    documents=apple_documents,
    embedding=embedding_model,
    collection_name="apple_knowledge_base"
)

print(f"✅ پایگاه برداری کروما با {len(apple_documents)} سند آماده‌سازی شد.")


### ۱. مقایسه تجربی: `Similarity Search` در برابر `MMR`

اکنون سوال دو وجهی کاربر را روی هر دو روش بازیابی اجرا می‌کنیم:
> **سوال:** *«اپل چطور تاسیس شد و محصولات اولیه‌اش چه بود؟»*


In [ ]:
# ۱. روش جستجوی شباهت سنتی (Similarity Search)
retriever_similarity = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

query = "اپل چطور تاسیس شد و محصولات اولیه‌اش چه بود؟"
docs_sim = retriever_similarity.invoke(query)

print("🔍 نتایج جستجوی شباهت سنتی (Similarity Search - k=3):")
print("=" * 60)
for idx, doc in enumerate(docs_sim, 1):
    print(f"[{idx}] (Topic: {doc.metadata['topic']}) -> {doc.page_content}")
print("=" * 60)
print("⚠️ مشاهده: هر ۳ سند بازیابی‌شده تکراری و صرفاً درباره تاسیس در گاراژ هستند!")
print("❌ اطلاعات مربوط به محصولات اولیه (Apple I و Apple II) کاملاً از دست رفت!")


In [ ]:
# ۲. روش بازیابی حداکثر ارتباط نهایی (MMR: Maximal Marginal Relevance)
retriever_mmr = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,             # تعداد نهایی اسناد مورد نیاز
        "fetch_k": 6,       # استخر اولیه کاندیداها
        "lambda_mult": 0.5  # ضریب تعادل طلایی بین شباهت و تنوع
    }
)

docs_mmr = retriever_mmr.invoke(query)

print("🎯 نتایج بازیابی با الگوریتم MMR (k=3, lambda=0.5):")
print("=" * 60)
for idx, doc in enumerate(docs_mmr, 1):
    print(f"[{idx}] (Topic: {doc.metadata['topic']}) -> {doc.page_content}")
print("=" * 60)
print("✅ پیروزی MMR: هم سند تاسیس در گاراژ و هم سند محصولات اولیه (Apple I & Apple II) بازیابی شدند!")


### ۲. کالبدشکافی پارامترهای MMR و اهرم تنظیم تنوع (`lambda_mult`)

الگوریتم MMR با این فرمول اسناد را امتیازدهی می‌کند:
$$\text{MMR} = \arg\max_{d_i \in C \setminus S} \left[ \lambda \cdot \text{Sim}(d_i, q) - (1 - \lambda) \cdot \max_{d_j \in S} \text{Sim}(d_i, d_j) \right]$$

پارامترهای کلیدی:
- **`fetch_k`**: اندازه استخر کاندیداها (مثلاً دیتابیس ابتدا ۲۰ سند با بیشترین شباهت را پیدا می‌کند).
- **`k`**: تعداد نهایی اسنادی که از میان استخر کاندیداها گلچین می‌شوند.
- **`lambda_mult` (بین 0.0 تا 1.0)**:
  - **`1.0`**: شباهت محض (دقیقاً معادل `similarity search`).
  - **`0.0`**: تنوع حداکثری (حتی به قیمت انتخاب اسناد کم‌ارتباط).
  - **`0.5`**: نقطه تعادل طلایی (Golden Balance) میان ارتباط به سوال و تنوع اطلاعاتی.

بیایید تاثیر مقادیر مختلف `lambda_mult` را به طور عینی مقایسه کنیم:


In [ ]:
lambda_values = [1.0, 0.5, 0.0]

for lm in lambda_values:
    custom_retriever = vector_db.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 6, "lambda_mult": lm}
    )
    res = custom_retriever.invoke(query)
    topics = [d.metadata['topic'] for d in res]
    print(f"📌 lambda_mult = {lm:<4} | موضوعات اسناد انتخاب‌شده: {topics}")
    for i, d in enumerate(res, 1):
        print(f"   {i}. [{d.metadata['topic']}] {d.page_content[:55]}...")
    print("-" * 50)


### ۳. حل مشکل سوالات پرت با آستانه اطمینان (`similarity_score_threshold`)

#### 🛑 معضل توهم در برابر سوالات بی‌ربط:
تصور کنید دیتابیس تخصصی قطعات خودرو یا فناوری دارید و کاربر ناگهان می‌پرسد:
> **«طرز تهیه قورمه‌سبزی چگونه است؟»**

در بازیابی عادی، سیستم موظف است حتماً $k$ سند را برگرداند! بنابراین مثلاً سند *«تعویض روغن موتور»* را چون کلمه «روغن» دارد برمی‌دارد و مدل در پرامپت سعی می‌کند با روغن موتور قورمه‌سبزی درست کند! (توهم فاجعه‌بار).

#### 🛡️ راه‌حل: `similarity_score_threshold`
به دیتابیس دستور می‌دهیم: **«اگر شباهت سندی کمتر از حداقل نمره اطمینان بود، اصلاً آن را نیاور!»**


In [ ]:
# تعریف بازیاب با آستانه نمره اطمینان
retriever_threshold = vector_db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.4}
)

# ۱. تست با سوال مرتبط:
rel_query = "اپل در چه سالی و کجا تاسیس شد؟"
rel_docs = retriever_threshold.invoke(rel_query)
print(f"✅ سوال مرتبط: '{rel_query}'")
print(f"تعداد اسناد بازیابی‌شده معتبر: {len(rel_docs)}")
for d in rel_docs:
    print(f"   -> {d.page_content}")

print()
print("=" * 60)
print()

# ۲. تست با سوال کاملاً بی‌ربط و پرت:
irrel_query = "طرز تهیه و پخت سنتی قورمه‌سبزی ایرانی چگونه است؟"
irrel_docs = retriever_threshold.invoke(irrel_query)
print(f"🛡️ سوال پرت و نامربوط: '{irrel_query}'")
print(f"تعداد اسناد بازیابی‌شده: {len(irrel_docs)}")
if len(irrel_docs) == 0:
    print("🎯 موفقیت: سیستم هیچ سندی بازیابی نکرد؛ مدل کانتکست خالی دریافت کرده و دچار توهم نمی‌شود!")


### ۴. ساخت خط لوله کامل RAG با الگوریتم MMR و مدل جمنای

حال زنجیره کامل RAG را با استفاده از **`retriever_mmr`** و مدل **`gemini-flash-lite-latest`** از طریق LCEL پیاده‌سازی و تست می‌کنیم:


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# راه‌اندازی مدل زبانی جمنای با کلید فعال محیط
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    api_key=os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY"),
    temperature=0
)

# پرامپت استاندارد مقیدساز RAG
template = """Answer the question based only on the following context:
{context}

Question: {question}
If you don't know the answer based on the context, simply say "I don't know". Do not make up an answer.
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

# ساخت زنجیره کامل RAG مجهز به بازیاب MMR
rag_chain_mmr = (
    {"context": retriever_mmr | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(f"✅ زنجیره کامل RAG با مدل '{llm.model}' و بازیاب MMR آماده شد.")


In [ ]:
# تست سوال چندوجهی با زنجیره MMR
query_multi = "اپل چطور تاسیس شد و محصولات اولیه‌اش چه بود؟"
response_mmr = rag_chain_mmr.invoke(query_multi)

print("💬 پاسخ جامع مدل جمنای با استفاده از زنجیره MMR:")
print("=" * 60)
print(response_mmr)
print("=" * 60)

# مقایسه با زنجیره ساده بدون MMR
rag_chain_simple = (
    {"context": retriever_similarity | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
response_simple = rag_chain_simple.invoke(query_multi)

print()
print("⚠️ پاسخ مدل بدون MMR (فقط Similarity Search ساده):")
print("=" * 60)
print(response_simple)
print("=" * 60)


### 📊 راهنمای تصمیم‌گیری (Decision Matrix): کدام الگوریتم بازیابی را انتخاب کنیم؟

| استراتژی بازیابی | نوع جستجو (`search_type`) | بهترین کاربرد | مزایا | معایب |
| :--- | :--- | :--- | :--- | :--- |
| **Similarity Search** | `similarity` | سوالات دقیق و فکت‌محور (مثلاً *«کد ملی علی چیست؟»* یا *«پایتخت فرانسه کجاست؟»*) | بسیار سریع، ساده و کم‌هزینه | تمایل شدید به آوردن چانک‌های تکراری و پر کردن بی‌مورد کانتکست |
| **MMR** | `mmr` | سوالات جامع، تحلیلی و چندوجهی (مثلاً *«تاریخچه شرکت و سبد محصولات آن چیست؟»*) | حذف اطلاعات افزونه و تضمین تنوع زوایای دید | نیاز به محاسبه فاصله بین اسناد کاندیدا (کمی بار محاسباتی بیشتر) |
| **Score Threshold** | `similarity_score_threshold` | سیستم‌های عمومی با ورودی‌های پیش‌بینی‌نشده برای دفاع در برابر سوالات نامربوط | جلوگیری قطعی از توهم در مواجهه با ورودی‌های پرت | نیاز به تنظیم دقیق مقدار threshold بر اساس مدل امبدینگ |

---

### 🔮 پیش‌درآمد درسنامه بعدی (تلفیق فیلترهای دقیق با جستجوی معنایی):
تا اینجا فرض کردیم سوال کاربر همیشه متنی و مفهومی است. اما اگر کاربر بپرسد:
> **«فیلم‌های علمی تخیلی سال ۲۰۲۰ را به من معرفی کن»**

جستجوی معنایی برداری مفهوم *«علمی تخیلی»* را می‌فهمد، اما عدد *«۲۰۲۰»* در فضای برداری به عنوان یک عدد ریاضی و دقیق رفتار نمی‌کند و ممکن است فیلم‌های ۲۰۱۹ یا ۲۰۲۱ را هم بیاورد!
در درسنامه بعد یاد می‌گیریم چگونه با استفاده از **فیلترهای متادیتا (Metadata Filtering و Self-Querying)** جستجوی معنایی را با شروط دقیق ترکیب کنیم.
